# Frame Extraction + DINO Distillation Pipeline
Run OpenCV-based frame extraction on a directory of unlabeled videos, then distill a YOLO pose model with DINOv3 teacher supervision in one notebook.

## How to use
1. Set the paths and distillation options in the config cell below.
2. Ensure OpenCV, numpy, tqdm, lightly, Ultralytics, and PyQt6 (for the directory picker) are installed in your `uv` environment.
3. Run the extraction section to populate `FRAME_OUTPUT_DIR` from your videos.
4. Run the distillation section to train YOLO on the extracted frames.


In [1]:
from pathlib import Path
from datetime import datetime

# ---- paths ----
PROJECT_ROOT = Path.cwd()
DISTILL_ROOT = PROJECT_ROOT / 'dino_distillations'
YOLO_YAML_CACHE = PROJECT_ROOT / 'yolo_yamls'
DISTILL_ROOT.mkdir(parents=True, exist_ok=True)
YOLO_YAML_CACHE.mkdir(parents=True, exist_ok=True)

# ---- extraction params ----
VIDEO_DIR = PROJECT_ROOT / 'videos' / 'new'  # directory of unlabeled mp4s
FRAME_OUTPUT_DIR = PROJECT_ROOT / 'data' / 'unlabeled_frames'
TOTAL_FRAMES_TO_EXTRACT = 500  # total number of frames to sample across all videos
CONCURRENT_STREAMS = 3

# ---- distillation params ----
YOLO_SIZE = 's'  # choose: 'n', 's', 'm', or 'l'
if YOLO_SIZE not in {'n', 's', 'm', 'l'}:
    raise ValueError(f"YOLO_SIZE must be one of {'n', 's', 'm', 'l'}; got {YOLO_SIZE!r}")
YOLO_MODEL = f'ultralytics/cfg/models/26/pose/yolo26{YOLO_SIZE}-pose.yaml'
FALLBACK_ULTRALYTICS_ALIAS = YOLO_MODEL
UNLABELED_DATA_DIR = FRAME_OUTPUT_DIR
TEACHER_ID = 'dinov3/vitb16'
TEACHER_WEIGHTS = PROJECT_ROOT / 'dinov3_vitb16_pretrain_lvd1689m-73cec8be.FSpy8ycZ.pth.part'

model_str = str(YOLO_MODEL)
model_fragment = Path(model_str).stem if model_str.endswith(('.yaml', '.yml')) else model_str.split('/')[-1]
MODEL_NAME = model_fragment  # e.g., yolo26n-pose
RUN_NAME = f"{MODEL_NAME}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUT_DIR = DISTILL_ROOT / RUN_NAME

EXPORTED_LAST_NAME = f'{MODEL_NAME}_last.pt'
CHECKPOINT_LAST_NAME = f'{MODEL_NAME}_last.ckpt'

print(f'Project root: {PROJECT_ROOT}')
print(f'Video dir: {VIDEO_DIR}')
print(f'Frame output dir: {FRAME_OUTPUT_DIR}')
print(f'Distillation output dir: {OUT_DIR}')


Project root: /home/neurodarden/Desktop/SqueakPoseStudio/dino_distillation/DINOv3_Distillation_YOLO-pose
Video dir: /home/neurodarden/Desktop/SqueakPoseStudio/dino_distillation/DINOv3_Distillation_YOLO-pose/videos/new
Frame output dir: /home/neurodarden/Desktop/SqueakPoseStudio/dino_distillation/DINOv3_Distillation_YOLO-pose/data/unlabeled_frames
Distillation output dir: /home/neurodarden/Desktop/SqueakPoseStudio/dino_distillation/DINOv3_Distillation_YOLO-pose/dino_distillations/yolo26s-pose_20260204_151802


### Select Video Directory
Run the cell below to pick the folder that contains your unlabeled `.mp4` files. If the dialog is dismissed, the path from the config cell is kept.

In [2]:
from pathlib import Path

VIDEO_DIR = Path(VIDEO_DIR)
selected_dir = None
try:
    from PyQt6 import QtWidgets
    app = QtWidgets.QApplication.instance() or QtWidgets.QApplication([])
    selected_dir = QtWidgets.QFileDialog.getExistingDirectory(
        None,
        "Select unlabeled video directory",
        str(VIDEO_DIR),
        QtWidgets.QFileDialog.Option.ShowDirsOnly,
    )
    if not QtWidgets.QApplication.instance().closingDown():
        app.quit()
except Exception as exc:
    print("PyQt6 directory picker unavailable. Set VIDEO_DIR manually in the config cell.")
    print(f"Reason: {exc}")
    selected_dir = None
if selected_dir:
    VIDEO_DIR = Path(selected_dir)
    print(f"Using video directory: {VIDEO_DIR}")
else:
    print(f"Directory selection canceled or unavailable; keeping existing path: {VIDEO_DIR}")


Using video directory: /home/neurodarden/Desktop


### Extraction Plan Preview
Inspect video fps/frame counts and preview how `TOTAL_FRAMES_TO_EXTRACT` will be distributed across videos before extraction.


In [3]:
from pathlib import Path

try:
    import cv2
except ImportError as exc:
    raise ImportError('Install opencv-python to preview extraction planning.') from exc


def allocate_frame_budget(frame_counts: list[int], total_target: int) -> list[int]:
    if total_target <= 0:
        raise ValueError('TOTAL_FRAMES_TO_EXTRACT must be > 0.')
    total_available = sum(frame_counts)
    if total_available <= 0:
        return [0] * len(frame_counts)

    target = min(total_target, total_available)
    raw = [target * c / total_available for c in frame_counts]
    base = [min(c, int(x)) for c, x in zip(frame_counts, raw)]
    remaining = target - sum(base)

    # Largest-remainder distribution with hard caps at available frames per video.
    order = sorted(
        range(len(frame_counts)),
        key=lambda i: (raw[i] - base[i], frame_counts[i]),
        reverse=True,
    )
    idx = 0
    while remaining > 0 and order:
        i = order[idx % len(order)]
        if base[i] < frame_counts[i]:
            base[i] += 1
            remaining -= 1
        idx += 1
        if idx > len(order) * max(1, target):
            break
    return base


VIDEO_DIR = Path(VIDEO_DIR)
video_paths = sorted(VIDEO_DIR.glob('*.mp4'))
if not video_paths:
    raise RuntimeError(f'No .mp4 files found in {VIDEO_DIR}')

metadata = []
print('Video summary:')
for path in video_paths:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f'Unable to open video: {path}')
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    duration = (total_frames / fps) if (total_frames > 0 and fps > 0) else 0.0
    cap.release()
    metadata.append({'path': path, 'frames': total_frames, 'fps': fps, 'duration': duration})
    fps_txt = f"{fps:.2f}" if fps else 'unknown'
    dur_txt = f"{duration:.1f}s" if duration else 'unknown'
    frames_txt = total_frames if total_frames else 'unknown'
    print(f" - {path.name}: duration {dur_txt} | fps {fps_txt} | frames {frames_txt}")

frame_counts = [m['frames'] for m in metadata]
total_available = sum(frame_counts)
if TOTAL_FRAMES_TO_EXTRACT <= 0:
    raise ValueError('TOTAL_FRAMES_TO_EXTRACT must be > 0.')

planned_total = min(TOTAL_FRAMES_TO_EXTRACT, total_available)
alloc = allocate_frame_budget(frame_counts, TOTAL_FRAMES_TO_EXTRACT)

print('\nExtraction plan:')
print(f' - Requested total frames: {TOTAL_FRAMES_TO_EXTRACT}')
print(f' - Total available frames: {total_available}')
print(f' - Planned extraction total: {planned_total}')
print(' - Per-video allocation:')
for m, n in zip(metadata, alloc):
    print(f"   * {m['path'].name}: {n} frames")


Video summary:
 - raw.mp4: duration 90268.2s | fps 30.00 | frames 2708149

Extraction plan:
 - Requested total frames: 500
 - Total available frames: 2708149
 - Planned extraction total: 500
 - Per-video allocation:
   * raw.mp4: 500 frames


### Set Total Frame Budget
Set a single total frame budget across all videos in `VIDEO_DIR`. Extraction allocates this budget proportionally by available frames per video.


In [4]:
# Optional one-off override for this session.
TOTAL_FRAMES_TO_EXTRACT = 500

if not isinstance(TOTAL_FRAMES_TO_EXTRACT, int) or TOTAL_FRAMES_TO_EXTRACT <= 0:
    raise ValueError('TOTAL_FRAMES_TO_EXTRACT must be a positive integer.')
print(f'Total frame budget set to: {TOTAL_FRAMES_TO_EXTRACT}')


Total frame budget set to: 500


In [5]:
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import cv2
from tqdm.auto import tqdm

FRAME_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def list_videos(video_root: Path):
    return sorted(video_root.glob('*.mp4'))


def check_video_io():
    print(f'OpenCV version: {cv2.__version__}')
    print(f'Saving extracted frames to: {FRAME_OUTPUT_DIR}')


def _write_frame(output_dir: Path, video_path: Path, frame, frame_idx: int):
    jpeg_params = [int(cv2.IMWRITE_JPEG_QUALITY), 100]
    frame_name = f"{video_path.stem}_idx{frame_idx:06d}.jpg"
    cv2.imwrite(str(output_dir / frame_name), frame, jpeg_params)


def allocate_frame_budget(frame_counts: list[int], total_target: int) -> list[int]:
    if total_target <= 0:
        raise ValueError('TOTAL_FRAMES_TO_EXTRACT must be > 0.')
    total_available = sum(frame_counts)
    if total_available <= 0:
        return [0] * len(frame_counts)

    target = min(total_target, total_available)
    raw = [target * c / total_available for c in frame_counts]
    base = [min(c, int(x)) for c, x in zip(frame_counts, raw)]
    remaining = target - sum(base)

    order = sorted(
        range(len(frame_counts)),
        key=lambda i: (raw[i] - base[i], frame_counts[i]),
        reverse=True,
    )
    idx = 0
    while remaining > 0 and order:
        i = order[idx % len(order)]
        if base[i] < frame_counts[i]:
            base[i] += 1
            remaining -= 1
        idx += 1
        if idx > len(order) * max(1, target):
            break
    return base


def extract_random_frames(video_path: Path, output_dir: Path, n_samples: int, position: int):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'Unable to open video: {video_path}')

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    if total_frames <= 0 or n_samples <= 0:
        cap.release()
        return 0

    n = min(n_samples, total_frames)
    rng = random.Random(hash(video_path.name) & 0xFFFFFFFF)
    indices = sorted(rng.sample(range(total_frames), n))

    progress = tqdm(total=len(indices), desc=f'{video_path.name}', unit='frame', position=position, leave=True)
    saved = 0
    try:
        for frame_idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ok, frame = cap.read()
            if not ok or frame is None:
                continue
            _write_frame(output_dir, video_path, frame, frame_idx)
            saved += 1
            progress.update(1)
    finally:
        cap.release()
        progress.close()

    return saved


def run_extraction(video_root: Path):
    videos = list_videos(video_root)
    if not videos:
        raise RuntimeError(f'No .mp4 files found under {video_root}')
    if not isinstance(TOTAL_FRAMES_TO_EXTRACT, int) or TOTAL_FRAMES_TO_EXTRACT <= 0:
        raise ValueError('TOTAL_FRAMES_TO_EXTRACT must be a positive integer.')

    check_video_io()

    frame_counts = []
    for vp in videos:
        cap = cv2.VideoCapture(str(vp))
        if not cap.isOpened():
            raise RuntimeError(f'Unable to open video: {vp}')
        frame_counts.append(int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0))
        cap.release()

    allocations = allocate_frame_budget(frame_counts, TOTAL_FRAMES_TO_EXTRACT)
    planned_total = sum(allocations)

    print(f'Processing {len(videos)} video(s) from {video_root}')
    print(f'Requested total frames: {TOTAL_FRAMES_TO_EXTRACT}')
    print(f'Planned extraction total: {planned_total}')

    summary = {}
    with ThreadPoolExecutor(max_workers=max(1, min(CONCURRENT_STREAMS, len(videos)))) as executor:
        futures = {
            executor.submit(extract_random_frames, vp, FRAME_OUTPUT_DIR, n_frames, idx): vp
            for idx, (vp, n_frames) in enumerate(zip(videos, allocations))
            if n_frames > 0
        }
        for future in as_completed(futures):
            vp = futures[future]
            saved = future.result()
            summary[vp.name] = saved
            print(f'[DONE] {vp.name}: {saved} frames')

    print('Extraction summary:')
    for name, count in summary.items():
        print(f' - {name}: {count} frames')
    print(f' - TOTAL: {sum(summary.values())} frames')
    return summary


/home/neurodarden/Desktop/SqueakPoseStudio/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import json
import logging
import os
import re
import shutil
from pathlib import Path

import lightly_train
import torch
import ultralytics
from ultralytics import YOLO

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'max_split_size_mb:64')
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


def _resolve_yaml_path(maybe_path: str) -> Path | None:
    suffix = Path(str(maybe_path)).suffix.lower()
    if suffix not in {'.yaml', '.yml'}:
        return None
    p = Path(maybe_path)
    if p.is_absolute() and p.is_file():
        return p
    candidate = (Path.cwd() / p).resolve()
    if candidate.is_file():
        return candidate
    return None


def _cache_ultralytics_yaml(alias: str) -> str | None:
    if 'ultralytics/' not in alias.lower():
        return None
    try:
        rel = alias.split('ultralytics/', 1)[1]
        package_root = Path(ultralytics.__file__).resolve().parent
        rel_path = Path(rel)
        search_root = package_root / 'cfg'
        candidates = []
        direct = package_root / rel_path
        if direct.is_file():
            candidates.append(direct)
        base_name = rel_path.name
        if not candidates:
            candidates.extend(search_root.rglob(base_name))
        if not candidates:
            simplified = re.sub(r'(yolo\d+)[nslmx](-)', r'\1\2', base_name)
            if simplified != base_name:
                candidates.extend(search_root.rglob(simplified))
        src = next((c for c in candidates if c.is_file()), None)
        if src is None:
            logging.warning('[Model] Unable to locate alias file inside Ultralytics package: %s', alias)
            return None
        YOLO_YAML_CACHE.mkdir(parents=True, exist_ok=True)
        dest = YOLO_YAML_CACHE / src.name
        shutil.copy2(src, dest)
        logging.info('[Model] Cached Ultralytics YAML to %s', dest)
        return str(dest)
    except Exception as exc:
        logging.warning('[Model] Failed to cache alias %s: %s', alias, exc)
        return None


def _autopatch_yaml(yaml_path: Path) -> Path:
    text = yaml_path.read_text(encoding='utf-8')
    had_trailing_newline = text.endswith('\n')
    if re.search(r'^\s*scale\s*:', text, flags=re.IGNORECASE | re.MULTILINE) is None:
        lines = text.splitlines()
        injected = False
        for i, line in enumerate(lines[:50]):
            if re.match(r'^\s*\w+\s*:', line):
                lines.insert(i + 1, 'scale: s')
                injected = True
                break
        if not injected:
            lines.insert(0, 'scale: s')
        text = '\n'.join(lines)
        if had_trailing_newline:
            text += '\n'
    text = re.sub(r'shortcut:\s*true', 'shortcut: false', text, flags=re.IGNORECASE)
    patched = yaml_path.with_suffix(
        yaml_path.suffix.replace('.yaml', '.autopatched.yaml').replace('.yml', '.autopatched.yml')
    )
    patched.write_text(text, encoding='utf-8')
    logging.warning('[Autopatch] Wrote patched YAML: %s', patched)
    return patched


def _write_scaled_yaml(base_yaml: Path, scale: str) -> Path:
    scale = str(scale).strip().lower()
    if scale not in {'n', 's', 'm', 'l'}:
        raise ValueError(f'Invalid scale {scale!r}; expected one of n/s/m/l.')

    text = base_yaml.read_text(encoding='utf-8')
    lines = text.splitlines()
    replaced = False
    for i, line in enumerate(lines):
        if re.match(r'^\s*scale\s*:', line):
            lines[i] = f'scale: {scale}'
            replaced = True
            break
    if not replaced:
        # Insert near top-level params section.
        insert_at = 0
        for i, line in enumerate(lines[:80]):
            if re.match(r'^\s*nc\s*:', line):
                insert_at = i + 1
                break
        lines.insert(insert_at, f'scale: {scale}')

    out = YOLO_YAML_CACHE / f'yolo26{scale}-pose.yaml'
    out.write_text('\n'.join(lines) + '\n', encoding='utf-8')
    logging.info('[Model] Wrote scaled YAML (%s) to %s', scale, out)
    return out


def _build_yolo_from_yaml_with_fallback(yaml_path: Path):
    try:
        logging.info('[Model] Using local YAML: %s', yaml_path)
        return YOLO(str(yaml_path))
    except AssertionError as exc:
        logging.warning('[Model] Assertion building YAML: %s', exc)
        logging.warning('[Model] Trying autopatched YAML...')
        patched = _autopatch_yaml(yaml_path)
        return YOLO(str(patched))


def _get_model_for_lightly(model_spec: str | Path):
    spec = str(model_spec)
    if spec.lower().startswith('ultralytics/'):
        # Lightly's Ultralytics wrapper may strip the package prefix and fail to resolve
        # custom aliases in some versions. Resolve to a real local YAML instead.
        local_base = PROJECT_ROOT / 'yolo26-pose.yaml'
        if local_base.is_file():
            scaled_yaml = _write_scaled_yaml(local_base, YOLO_SIZE)
            return _build_yolo_from_yaml_with_fallback(scaled_yaml)
        logging.warning('[Model] Local base YAML missing (%s); passing alias directly.', local_base)
        return spec

    yaml_path = _resolve_yaml_path(spec)
    if yaml_path is not None:
        try:
            return _build_yolo_from_yaml_with_fallback(yaml_path)
        except Exception as exc:
            logging.warning('[Model] Local YAML build failed (%s); falling back to alias.', exc)
            return FALLBACK_ULTRALYTICS_ALIAS

    if spec.lower().endswith(('.yaml', '.yml')):
        logging.warning('[Model] YAML path missing (%s); falling back to alias.', spec)
        return FALLBACK_ULTRALYTICS_ALIAS

    return spec


def _sanity_probe_ultralytics_head(model_obj_or_spec):
    try:
        m = model_obj_or_spec if isinstance(model_obj_or_spec, YOLO) else YOLO(model_obj_or_spec)
        task = getattr(m, 'task', 'unknown')
        head_cls = type(m.model.model[-1]).__name__ if hasattr(m.model, 'model') and m.model.model else 'unknown'
        logging.info('[Sanity] Ultralytics parsed task=%s | head=%s', task, head_cls)
        head_names = [n for n, _ in m.model.named_modules()]
        has_dfl = any('.dfl' in n for n in head_names)
        if task != 'pose':
            logging.warning('[Sanity] Model task=%s (expected pose). Confirm YOLO_MODEL.', task)
        elif has_dfl:
            logging.info('[Sanity] DFL module detected with pose head (%s). This is acceptable for YOLO26 pose.', head_cls)
    except Exception as exc:
        logging.warning('[Sanity] Could not probe Ultralytics model: %s', exc)


def _assert_model_scale(model_spec: str | Path, expected_size: str):
    size = str(expected_size).strip().lower()
    if size not in {'n', 's', 'm', 'l'}:
        raise ValueError(f'Invalid YOLO_SIZE={expected_size!r}; expected one of n/s/m/l.')
    spec = str(model_spec)
    target = f'yolo26{size}-pose'
    if target not in spec:
        raise ValueError(f'Scale mismatch: expected model containing {target!r}, got {spec!r}')
    logging.info('[Scale] Using %s for distillation', target)


def _runtime_preflight():
    logging.info('[Runtime] torch=%s | torch.cuda=%s | ultralytics=%s | lightly_train=%s',
                 torch.__version__,
                 torch.version.cuda,
                 getattr(ultralytics, '__version__', 'unknown'),
                 getattr(lightly_train, '__version__', 'unknown'))
    if not torch.cuda.is_available():
        raise EnvironmentError('CUDA is not available. Distillation requires a CUDA GPU in this notebook setup.')
    device_idx = 0
    gpu_name = torch.cuda.get_device_name(device_idx)
    capability = torch.cuda.get_device_capability(device_idx)
    logging.info('[Runtime] CUDA device[%d]=%s | capability=%s', device_idx, gpu_name, capability)
    compiled_cuda = torch.version.cuda
    if compiled_cuda:
        try:
            comp_major, comp_minor = [int(x) for x in compiled_cuda.split('.')[:2]]
            if (capability[0], capability[1]) > (comp_major, comp_minor):
                logging.warning('[Runtime] GPU capability %s is newer than compiled CUDA %s. Continue if stable.', capability, compiled_cuda)
        except Exception:
            pass
    _ = (torch.tensor([1.0], device='cuda') * 2).item()
    torch.cuda.synchronize()
    logging.info('[Runtime] CUDA tensor smoke test passed')


def _teacher_weights_path() -> Path:
    p = Path(TEACHER_WEIGHTS).expanduser().resolve()
    if not p.is_file():
        raise FileNotFoundError(f'Teacher weights not found: {p}')
    if p.stat().st_size == 0:
        raise RuntimeError(f'Teacher weights file is empty (0 bytes): {p}')
    return p


def _method_args():
    teacher = _teacher_weights_path()
    return {
        'teacher': TEACHER_ID,
        'teacher_url': teacher.as_uri(),
    }


def run_distillation(epochs: int | None = None, batch_size: int | None = None):
    unlabeled_dir = Path(UNLABELED_DATA_DIR)
    if not unlabeled_dir.is_dir():
        raise FileNotFoundError(f'UNLABELED_DATA_DIR missing: {unlabeled_dir}')

    image_exts = ('*.jpg', '*.jpeg', '*.png')
    image_count = sum(len(list(unlabeled_dir.glob(ext))) for ext in image_exts)
    if image_count == 0:
        raise RuntimeError(f'No image frames found in {unlabeled_dir}. Run extraction first or update UNLABELED_DATA_DIR.')
    logging.info('[Data] Using %d images from %s', image_count, unlabeled_dir)

    if epochs is not None and (not isinstance(epochs, int) or epochs <= 0):
        raise ValueError(f'epochs must be a positive int or None, got {epochs!r}')
    if batch_size is not None and (not isinstance(batch_size, int) or batch_size <= 0):
        raise ValueError(f'batch_size must be a positive int or None, got {batch_size!r}')

    _runtime_preflight()
    _assert_model_scale(YOLO_MODEL, YOLO_SIZE)

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    model_for_lightly = _get_model_for_lightly(YOLO_MODEL)
    _sanity_probe_ultralytics_head(model_for_lightly)

    log_args = {
        'out': str(OUT_DIR),
        'data': str(unlabeled_dir),
        'image_count': image_count,
        'model': str(YOLO_MODEL),
        'resolved_model_type': type(model_for_lightly).__name__,
        'epochs_override': epochs,
        'batch_size_override': batch_size,
        'note': 'Using Lightly pretrain defaults unless optional overrides are provided',
    }
    logging.info('[Lightly] Launch args: %s', json.dumps(log_args, indent=2))

    pretrain_kwargs = {
        'out': str(OUT_DIR),
        'data': str(unlabeled_dir),
        'model': model_for_lightly,
        'method': 'distillation',
        'method_args': _method_args(),
    }
    if epochs is not None:
        pretrain_kwargs['epochs'] = epochs
    if batch_size is not None:
        pretrain_kwargs['batch_size'] = batch_size

    lightly_train.pretrain(**pretrain_kwargs)

    export_dir = OUT_DIR / 'exported_models'
    ckpt_dir = OUT_DIR / 'checkpoints'
    export_dir.mkdir(parents=True, exist_ok=True)

    exported_last = export_dir / 'exported_last.pt'
    last_ckpt = ckpt_dir / 'last.ckpt'
    if not exported_last.exists():
        if not last_ckpt.exists():
            raise FileNotFoundError(f'Export and checkpoint missing under {OUT_DIR}')
        logging.info('[Lightly] Export missing, exporting from last.ckpt ...')
        lightly_train.export(
            out=str(exported_last),
            checkpoint=str(last_ckpt),
            part='model',
            format='package_default',
        )

    named_last_export = export_dir / EXPORTED_LAST_NAME
    if exported_last.exists() and named_last_export != exported_last:
        exported_last.replace(named_last_export)

    if last_ckpt.exists():
        shutil.copy2(last_ckpt, ckpt_dir / CHECKPOINT_LAST_NAME)

    logging.info('[Lightly] Exported model at %s', named_last_export)
    return named_last_export


### Run Frame Extraction
Run this after setting `TOTAL_FRAMES_TO_EXTRACT`. The frame budget is distributed across all videos and sampled randomly within each video.


In [7]:
print('=== Running frame extraction ===')
extraction_summary = run_extraction(VIDEO_DIR)
extraction_summary


=== Running frame extraction ===
OpenCV version: 4.13.0
Saving extracted frames to: /home/neurodarden/Desktop/SqueakPoseStudio/dino_distillation/DINOv3_Distillation_YOLO-pose/data/unlabeled_frames
Processing 1 video(s) from /home/neurodarden/Desktop
Requested total frames: 500
Planned extraction total: 500


raw.mp4: 100%|██████████| 500/500 [00:06<00:00, 72.40frame/s]

[DONE] raw.mp4: 500 frames
Extraction summary:
 - raw.mp4: 500 frames
 - TOTAL: 500 frames


{'raw.mp4': 500}

### Run Distillation
Launch Lightly distillation once frames are ready.


In [8]:
print('=== Running distillation ===')
DISTILL_EPOCHS = 10      # set e.g. 50 to override Lightly default
DISTILL_BATCH_SIZE = None  # set e.g. 512 to override Lightly default (default 128)
exported_checkpoint = run_distillation(epochs=DISTILL_EPOCHS, batch_size=DISTILL_BATCH_SIZE)
print(f'Model exported to: {exported_checkpoint}')
exported_checkpoint


2026-02-04 15:18:22,397 - INFO - [Data] Using 500 images from /home/neurodarden/Desktop/SqueakPoseStudio/dino_distillation/DINOv3_Distillation_YOLO-pose/data/unlabeled_frames
2026-02-04 15:18:22,397 - INFO - [Runtime] torch=2.10.0+cu130 | torch.cuda=13.0 | ultralytics=8.4.11 | lightly_train=0.14.0
2026-02-04 15:18:22,452 - WARNING - /home/neurodarden/Desktop/SqueakPoseStudio/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()

2026-02-04 15:18:22,454 - INFO - [Runtime] CUDA device[0]=NVIDIA GB10 | capability=(12, 1)


=== Running distillation ===


2026-02-04 15:18:22,648 - INFO - [Runtime] CUDA tensor smoke test passed
2026-02-04 15:18:22,648 - INFO - [Scale] Using yolo26s-pose for distillation
2026-02-04 15:18:22,649 - INFO - [Model] Wrote scaled YAML (s) to /home/neurodarden/Desktop/SqueakPoseStudio/dino_distillation/DINOv3_Distillation_YOLO-pose/yolo_yamls/yolo26s-pose.yaml
2026-02-04 15:18:22,649 - INFO - [Model] Using local YAML: /home/neurodarden/Desktop/SqueakPoseStudio/dino_distillation/DINOv3_Distillation_YOLO-pose/yolo_yamls/yolo26s-pose.yaml
2026-02-04 15:18:22,763 - INFO - [Sanity] Ultralytics parsed task=pose | head=Pose26
2026-02-04 15:18:22,764 - INFO - [Sanity] DFL module detected with pose head (Pose26). This is acceptable for YOLO26 pose.
2026-02-04 15:18:22,764 - INFO - [Lightly] Launch args: {
  "out": "/home/neurodarden/Desktop/SqueakPoseStudio/dino_distillation/DINOv3_Distillation_YOLO-pose/dino_distillations/yolo26s-pose_20260204_151802",
  "data": "/home/neurodarden/Desktop/SqueakPoseStudio/dino_distillat

Epoch 0: 100%|██████████| 3/3 [00:03<00:00,  0.91it/s, train_loss=0.403, data_wait=3.8%]

2026-02-04 15:18:26,642 - DEBUG - Exporting model to '/home/neurodarden/Desktop/SqueakPoseStudio/dino_distillation/DINOv3_Distillation_YOLO-pose/dino_distillations/yolo26s-pose_20260204_151802/exported_models/exported_last.pt' in format 'ModelFormat.PACKAGE_DEFAULT'.


...

2026-02-04 15:19:00,273 - DEBUG - Exporting model to '/home/neurodarden/Desktop/SqueakPoseStudio/dino_distillation/DINOv3_Distillation_YOLO-pose/dino_distillations/yolo26s-pose_20260204_151802/exported_models/exported_last.pt' in format 'ModelFormat.PACKAGE_DEFAULT'.
`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 3/3 [00:04<00:00,  0.75it/s, train_loss=0.193, data_wait=2.6%]


Training completed.
2026-02-04 15:19:03,046 - INFO - Training completed.
2026-02-04 15:19:03,047 - DEBUG - Exporting model to '/home/neurodarden/Desktop/SqueakPoseStudio/dino_distillation/DINOv3_Distillation_YOLO-pose/dino_distillations/yolo26s-pose_20260204_151802/exported_models/exported_last.pt' in format 'ModelFormat.PACKAGE_DEFAULT'.
Example: How to use the exported model
----------------------------------------------------------------------------------------
from ultralytics import YOLO

# Load the pretrained model
model = YOLO('/home/neurodarden/Desktop/SqueakPoseStudio/dino_distillation/DINOv3_Distillation_YOLO-pose/dino_distillations/yolo26s-pose_20260204_151802/exported_models/exported_last.pt')

# Finetune or evaluate the model
...
----------------------------------------------------------------------------------------

2026-02-04 15:19:03,177 - INFO - Example: How to use the exported model
-------------------------------------------------------------------------------------

Model exported to: /home/neurodarden/Desktop/SqueakPoseStudio/dino_distillation/DINOv3_Distillation_YOLO-pose/dino_distillations/yolo26s-pose_20260204_151802/exported_models/yolo26s-pose_last.pt


PosixPath('/home/neurodarden/Desktop/SqueakPoseStudio/dino_distillation/DINOv3_Distillation_YOLO-pose/dino_distillations/yolo26s-pose_20260204_151802/exported_models/yolo26s-pose_last.pt')

In [9]:
from pathlib import Path
from ultralytics import YOLO
import torch

NOTEBOOK_ROOT = Path.cwd()
RUNS_ROOT = NOTEBOOK_ROOT / "dino_distillations"

candidates = sorted(
    RUNS_ROOT.glob("*/exported_models/exported_last.pt"),
    key=lambda p: p.stat().st_mtime,
)
if not candidates:
    raise FileNotFoundError(f"No exported_last.pt found under {RUNS_ROOT}")

EXPORT_PT = candidates[-1]
print(f"Using export: {EXPORT_PT}")

model = YOLO(str(EXPORT_PT))
nn_model = model.model
nn_model.eval()

print(f"File size: {EXPORT_PT.stat().st_size / (1024**2):.2f} MB")
print(f"Task: {model.task}")
print(f"Head: {type(nn_model.model[-1]).__name__ if hasattr(nn_model, 'model') else 'unknown'}")

total_params = sum(p.numel() for p in nn_model.parameters())
trainable_params = sum(p.numel() for p in nn_model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")


Using export: /home/neurodarden/Desktop/SqueakPoseStudio/dino_distillation/DINOv3_Distillation_YOLO-pose/dino_distillations/yolo26s-pose_20260204_151802/exported_models/exported_last.pt
File size: 23.10 MB
Task: pose
Head: Pose26
Total params: 11,870,498
Trainable params: 11,870,498


In [10]:
from pathlib import Path
from ultralytics import YOLO

NOTEBOOK_ROOT = Path.cwd()
RUNS_ROOT = NOTEBOOK_ROOT / "dino_distillations"  # no extra segments needed
exports = sorted(RUNS_ROOT.glob("*/exported_models/exported_last.pt"),
                 key=lambda p: p.stat().st_mtime)

if not exports:
    raise FileNotFoundError(f"No exported_last.pt found under {RUNS_ROOT}")

pt = exports[-1]
print(f"Using: {pt}")

m = YOLO(str(pt))
scale = None
try:
    scale = m.model.yaml.get("scale")
except Exception:
    pass

print("Task:", m.task)
print("Scale from yaml:", scale if scale else "unknown")


Using: /home/neurodarden/Desktop/SqueakPoseStudio/dino_distillation/DINOv3_Distillation_YOLO-pose/dino_distillations/yolo26s-pose_20260204_151802/exported_models/exported_last.pt
Task: pose
Scale from yaml: s


In [11]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

NOTEBOOK_ROOT = Path.cwd()
RUNS_ROOT = NOTEBOOK_ROOT / "dino_distillations"
RUN_DIR = max((p for p in RUNS_ROOT.iterdir() if p.is_dir()), key=lambda p: p.stat().st_mtime)
METRICS_PATH = RUN_DIR / "metrics.jsonl"
OUT_PATH = RUN_DIR / "train_loss_curve.png"

records = []
with METRICS_PATH.open("r", encoding="utf-8") as fh:
    for line in fh:
        if not line.strip():
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue

        loss_val = rec.get("loss")
        if loss_val is None:
            loss_val = rec.get("train_loss")
        if loss_val is None:
            continue

        records.append({
            "epoch": rec.get("epoch", rec.get("step")),
            "loss": float(loss_val),
        })

if not records:
    raise RuntimeError("No loss or train_loss entries found in metrics.jsonl")

df = pd.DataFrame(records).sort_values("epoch")

plt.figure(figsize=(7, 4))
plt.plot(df["epoch"], df["loss"], marker="o")
plt.title("DINO Distillation – Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_PATH, dpi=200)
plt.close()

print(f"Saved loss curve to {OUT_PATH}")


Saved loss curve to /home/neurodarden/Desktop/SqueakPoseStudio/dino_distillation/DINOv3_Distillation_YOLO-pose/dino_distillations/yolo26s-pose_20260204_151802/train_loss_curve.png
